In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, os, time, json, re, csv, tqdm, spacy
from glob import glob

from sklearn.metrics import classification_report

from collections import defaultdict
from pathlib import Path

sys.path.append('/Proyecto/Value-disagreement/Python/Utilities')
#import text_cleansing, Dict_Object

In [2]:
#!pip install spellchecker

### Value Dictionary

In [3]:
### DEFINIMOS UNA VARIABLE QUE NOS FACILITE LA RUTA DE LOS ARCHIVOS
#program_path = os.path.abspath(os.path.dirname("__file__"))
program_path = "/Proyecto/Value-disagreement/Datos/Dictionary/osfstorage-archive"
print(program_path)

/home/varsayou/Desktop/Clases/Proyecto/Value-disagreement/Datos/Dictionary/osfstorage-archive


In [4]:
def ObtenerArchivos(ruta_actual,carpeta):
    """Esta función nos devuelve una lista con los archivos de una carpeta"""
    ruta_completa = os.path.join(ruta_actual,carpeta)
    archivos = glob(ruta_completa+"/*")
    return archivos

In [5]:
files = ObtenerArchivos(program_path,'Dictionaries')
list(files)

['/home/varsayou/Desktop/Clases/Proyecto/Value-disagreement/Datos/Dictionary/osfstorage-archive/Dictionaries/Provisional_dictionary.txt',
 '/home/varsayou/Desktop/Clases/Proyecto/Value-disagreement/Datos/Dictionary/osfstorage-archive/Dictionaries/Refined_dictionary.txt']

In [6]:
data = {'value_id':['1','2','3','4','5','6','7','8','9','10'],
        'value_short':['SE','CO','TR','BE','UN','SD','ST','HE','AC','PO'],
        'value_long':['SECURITY','CONFORMITY','TRADITION','BENEVOLENCE','UNIVERSALISM',
                      'SELF-DIRECTION','STIMULATION','HEDONISM','ACHIEVEMENT','POWER']}

data_df = pd.DataFrame.from_dict(data, dtype='object')
data_df

,value_id,value_short,value_long
0,1,SE,SECURITY
1,2,CO,CONFORMITY
2,3,TR,TRADITION
3,4,BE,BENEVOLENCE
4,5,UN,UNIVERSALISM
5,6,SD,SELF-DIRECTION
6,7,ST,STIMULATION
7,8,HE,HEDONISM
8,9,AC,ACHIEVEMENT
9,10,PO,POWER


In [7]:
lista_txt = []
for archivo in files:
    print("archivo: ",archivo)
    df = pd.read_csv(archivo, sep="\t", names=['word','value_id'])
    df["ID_FILE"] = archivo
    df[['Directorio','Archivo']] = df["ID_FILE"].astype(str).str.split("Dictionaries",expand = True)
    df["dic_type"] = df["Archivo"].astype(str).str.strip("/").str.strip("*\.txt").str.strip("_dictionary")
    df["value_id"] = df["value_id"].astype(str).str.strip("'")
    df.drop(columns=['ID_FILE', 'Directorio', 'Archivo'], inplace=True)
    lista_txt.append(df)
        
dics = pd.concat(lista_txt,axis=0, sort = False)
dics.reset_index(inplace=True,drop=True)

value_dic = dics.merge(data_df, on='value_id')
value_dic.drop_duplicates(subset='word',inplace=True, keep='last', ignore_index=True)

value_dic

archivo:  /home/varsayou/Desktop/Clases/Proyecto/Value-disagreement/Datos/Dictionary/osfstorage-archive/Dictionaries/Provisional_dictionary.txt
archivo:  /home/varsayou/Desktop/Clases/Proyecto/Value-disagreement/Datos/Dictionary/osfstorage-archive/Dictionaries/Refined_dictionary.txt


<>:7: SyntaxWarning: invalid escape sequence '\.'
<>:7: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_23023/3025579202.py:7: SyntaxWarning: invalid escape sequence '\.'
  df["dic_type"] = df["Archivo"].astype(str).str.strip("/").str.strip("*\.txt").str.strip("_dictionary")


,word,value_id,dic_type,value_short,value_long
0,aegis,1,Provisional,SE,SECURITY
1,alertness,1,Provisional,SE,SECURITY
2,believability,1,Provisional,SE,SECURITY
3,bulwark,1,Provisional,SE,SECURITY
4,canniness,1,Provisional,SE,SECURITY
...,...,...,...,...,...
2572,weaker,10,Refine,PO,POWER
2573,weakness,10,Refine,PO,POWER
2574,weaknesses,10,Refine,PO,POWER
2575,wealth,10,Refine,PO,POWER


### ValueNet Dictionary

In [8]:
valuenet_dic = pd.read_excel(r'/Proyecto/Value-disagreement/Datos/ValueNET/valunet_dic.xlsx')
valuenet_dic = valuenet_dic.merge(data_df, left_on='value', right_on='value_long')
valuenet_dic.rename(columns={"valunet_set": "dic_type"}, inplace=True)
valuenet_dic.drop(columns=['value'], inplace=True)
valuenet_dic = valuenet_dic[['word','value_id','dic_type','value_short','value_long']]
valuenet_dic

,word,value_id,dic_type,value_short,value_long
0,healthy,1,original,SE,SECURITY
1,family,1,original,SE,SECURITY
2,order,1,original,SE,SECURITY
3,clean,1,original,SE,SECURITY
4,safety,1,original,SE,SECURITY
...,...,...,...,...,...
170,buddhist,3,datamuse_keyword,TR,TRADITION
171,republican,3,datamuse_keyword,TR,TRADITION
172,islamic,3,datamuse_keyword,TR,TRADITION
173,responsibility,3,glove_keyword,TR,TRADITION


### All in One Dic

In [9]:
all_dic = pd.concat([value_dic,valuenet_dic],axis=0, sort = False) ## 2733 rows
all_dic.drop_duplicates(subset='word',inplace=True, keep='first', ignore_index=True)
all_dic

,word,value_id,dic_type,value_short,value_long
0,aegis,1,Provisional,SE,SECURITY
1,alertness,1,Provisional,SE,SECURITY
2,believability,1,Provisional,SE,SECURITY
3,bulwark,1,Provisional,SE,SECURITY
4,canniness,1,Provisional,SE,SECURITY
...,...,...,...,...,...
2617,moderate,3,original,TR,TRADITION
2618,pious,3,datamuse_keyword,TR,TRADITION
2619,classic,3,datamuse_keyword,TR,TRADITION
2620,christian,3,datamuse_keyword,TR,TRADITION


In [11]:
all_dic.to_csv(r"/Proyecto/Value-disagreement/Datos/Dictionary/osfstorage-archive/Dictionaries/value_words_dict.csv",
              header=True, index=False, quoting=csv.QUOTE_ALL, quotechar='"', sep="|", escapechar="|"
              )

### VALUENET

In [ ]:
valuenet = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_all_10.csv",
                         sep='|')
valuenet

In [ ]:
valuenet[valuenet.duplicated(subset=['value','scenario'])]

In [ ]:
valuenet_pivot = valuenet.pivot(index='scenario', columns='value', values='label').fillna(0).astype(int).reset_index()
valuenet_pivot

### VALUEARG

In [ ]:
valuearg = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_all.csv",
                       sep='|')
valuearg.rename(columns={"Argument ID": "uid","variable":"value","value":"label","Premise":"scenario"}, inplace=True)
valuearg.drop_duplicates(subset=['value','scenario'],inplace=True, keep='first', ignore_index=True)
valuearg

In [ ]:
valuearg[valuearg.duplicated(subset=['value','scenario'],keep=False)]

In [ ]:
valuearg_pivot = valuearg.pivot(index='scenario', columns='value', values='label').fillna(0).astype(int).reset_index()
valuearg_pivot

In [ ]:
valuearg = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_all_10.csv",
                         sep='|')
valuearg

### All (valuenet+valuearg)

In [ ]:
value_all = pd.concat([valuenet,valuearg],axis=0, sort = False) ## 2733 rows
#all_dic.drop_duplicates(subset='word',inplace=True, keep='first', ignore_index=True)
value_all

In [ ]:
value_all['scenario_cleaned'] = value_all['scenario'].apply(text_cleansing.clean_text)
value_all

In [ ]:
value_all[value_all.duplicated(subset=['value','scenario'],keep=False)]

In [ ]:
52690+21374

In [ ]:
value_all = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_10.csv",
                         sep='|')
value_all

### BaseLine Dic Classifier

In [ ]:
def get_dataset(dataset_name):
    """
    Load dataset based on name.
    """
    if dataset_name == "valueeval":
        dataset = ValueEvalDataset(
            "~/projects/implicit/aspects/data/valueeval/dataset-identifying-the-human-values-behind-arguments/",
            cast_to_valuenet=True,
            return_predefined_splits=True
        )
    elif dataset_name == "valuenet":
        dataset = ValueNetDataset("~/projects/implicit/aspects/data/valuenet/", return_predefined_splits=True)
    else:
        dataset = RedditAnnotatedDataset(dataset_name)
    return dataset

In [ ]:
def get_splits(df, train_size=0.8, val_size=0.1, test_size=0.1):
    assert train_size + val_size + test_size == 1
    
    num_train = int(len(df)*train_size)
    num_val = int(len(df)*val_size)
    num_test = len(df) - (num_train + num_val)

    idx = np.arange(len(df))
    np.random.shuffle(idx)
    train_set = idx[:num_train]
    val_set = idx[num_train:num_train + num_val]
    test_set = idx[num_train + num_val:]
    
    return train_set, val_set, test_set

In [ ]:
valuenet_train_set, valuenet_val_set, valuenet_test_set = get_splits(valuenet)
valuenet_train_set.tofile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_train_10.csv",sep=',', format='%s')
valuenet_val_set.tofile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_val_10.csv",sep=',', format='%s')
valuenet_test_set.tofile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_test_10.csv",sep=',', format='%s')

#valuenet_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_train.csv",sep=',')
#valuenet_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_val.csv",sep=',')
#valuenet_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_test.csv",sep=',')
#valuenet_test_set

In [ ]:
valuearg_train_set, valuearg_val_set, valuearg_test_set = get_splits(valuearg)
valuearg_train_set.tofile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_train_10.csv",sep=',', format='%s')
valuearg_val_set.tofile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_val_10.csv",sep=',', format='%s')
valuearg_test_set.tofile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_test_10.csv",sep=',', format='%s')

#valuearg_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_train.csv",sep=',')
#valuearg_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_val.csv",sep=',')
#valuearg_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_test.csv",sep=',')
#valuearg_val_set

In [ ]:
value_all_train_set, value_all_val_set, value_all_test_set = get_splits(value_all)
value_all_train_set.tofile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train_10.csv",sep=',', format='%s')
value_all_val_set.tofile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val_10.csv",sep=',', format='%s')
value_all_test_set.tofile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test_10.csv",sep=',', format='%s')

#value_all_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv",sep=',')
#value_all_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv",sep=',')
#value_all_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv",sep=',')
#value_all_val_set

In [ ]:
def classify_comment_value(dic, comment, scoring="any", min_coun=0):
    # Optionally preprocess comment
    #if 'lemmatize' in self.preprocessing:
    #    doc = self.nlp(comment)
    #    " ".join([token.lemma_ for token in doc])

    values = dic['value_long'].unique()
    
    # Split on spaces
    split_comment = comment.split()

    labels = []
    for value in values:
        words = dic['word'].unique()
        
        if scoring == "any":
            value_present = any([(word in split_comment) for word in words])
            if value_present:
                labels.append(value)
                
        elif scoring == "min_count":
            words_present = [(word in split_comment) for word in words]
            if sum(words_present) >= min_count:
                labels.append(value)
                
        elif scoring == "value_score":
            for word in words:
                if word in split_comment:
                    labels.append(value)

    return labels

In [ ]:
def get_results(dataset, dic, test_idx, cleaned=False):
    """
    Get predictions using the value dictionary.
    """
    y_true = []
    y_pred_vd = []
    for i in test_idx:
        label = dataset.iloc[int(i)]['label']
        if cleaned == True:
            text = dataset.iloc[int(i)]['scenario_cleaned']
        else:
            text = dataset.iloc[int(i)]['scenario']
        dictionary_pred = classify_comment_value(dic,text)
        value = dataset.iloc[int(i)]['value']
        y_pred_vd.append(int(value in dictionary_pred))
        y_true.append(abs(label))

    return y_pred_vd, y_true

In [ ]:
#y_pred_vd, y_true = get_results(valuenet, all_dic, valuenet_train_set)

In [ ]:
runs= {'valuenet':(valuenet, all_dic, valuenet_train_set),
       'valuearg':(valuearg, all_dic, valuearg_train_set),
       'value_all':(value_all, all_dic, value_all_train_set),
       'value_all_cleaned':(value_all, all_dic, value_all_train_set)}

In [ ]:
def print_results(dataset, y_true, y_pred, test_idx, all_ones=False):
    """
    Print the results.
    """
    res_df = pd.DataFrame()
    if all_ones == True:
        print("All ones")
        y_pred_all_ones = np.ones((len(test_idx)))
        report_dict = classification_report(y_true, y_pred_all_ones, zero_division=0, output_dict=True)
        df_report = pd.DataFrame(report_dict).transpose()
        df_report['dataset'] = "all-ones"
        res_df = pd.concat([res_df, df_report], ignore_index=True)
        print(df_report)

    print("Value dictionary: both,  Set: {}".format(dataset))
    report_dict = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    df_report = pd.DataFrame(report_dict).transpose()
    df_report['dataset'] = dataset
    res_df = pd.concat([res_df, df_report], ignore_index=True)
    print(df_report)
    return res_df

In [ ]:
results_df = pd.DataFrame()
flag = 0
for key, item in runs.items():
    if key == 'value_all_cleaned':
        if flag == 0:
            y_pred_vd, y_true = get_results(item[0], item[1], item[2], True)
            df = print_results(key, y_true, y_pred_vd, item[2], True)
            results_df = pd.concat([results_df, df], ignore_index=True)
            flag = 1
        else:
            y_pred_vd, y_true = get_results(item[0], item[1], item[2], True)
            df = print_results(key, y_true, y_pred_vd, item[2])
            results_df = pd.concat([results_df, df], ignore_index=True)
            
    if flag == 0:
        y_pred_vd, y_true = get_results(item[0], item[1], item[2])
        df = print_results(key, y_true, y_pred_vd, item[2], True)
        results_df = pd.concat([results_df, df], ignore_index=True)
        flag = 1
    else:
        y_pred_vd, y_true = get_results(item[0], item[1], item[2])
        df = print_results(key, y_true, y_pred_vd, item[2])
        results_df = pd.concat([results_df, df], ignore_index=True)
results_df

In [ ]:
results_df.to_csv(r"/Proyecto/Value-disagreement/Datos/dict_eval_metrics.csv",sep=',')